# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- **License**: [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show the dataset name and description
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id` values, and fields. All entities are referenced by their `@id` as defined in the Croissant metadata.

**Let's enumerate record sets, the structure, and some example records from the main data table.**

In [ ]:
# List available record sets and their fields
print("Available Record Sets:")
record_sets = dataset.metadata.record_sets
for idx, rset in enumerate(record_sets):
    print(f"{idx+1}. @id: {rset.id}")
    print(f"   Name: {rset.name}")
    print(f"   Description: {rset.description}")
    if hasattr(rset, 'fields'):
        print(f"   Fields:")
        for field in rset.fields:
            print(f"    - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', 'N/A')}")
    print()

# For illustration, let's preview the first record set's first few records by @id.
if len(record_sets) > 0:
    example_record_set_id = record_sets[0].id
    print(f"\nPreviewing records from Record Set @id: {example_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        if i>=3:
            break
        print(record)

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Reference each record set and field using their `@id` fields as identified above.

We'll construct a dictionary mapping each record set's `@id` to its corresponding DataFrame.

In [ ]:
# Extract data from all record sets (using @id)
dataframes = {}
for rset in dataset.metadata.record_sets:
    recs = list(dataset.records(record_set=rset.id))
    # If there are records, store in DataFrame
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rset.id] = df
        print(f"Loaded {len(df)} records for Record Set @{rset.id} (columns: {df.columns.tolist()})\n")

# For demonstration, display columns and preview first 5 rows from the main record set
main_record_set_id = list(dataframes.keys())[0]  # Use the first one as main
print(f"Columns in Record Set @{main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Now we perform some standard EDA: filter numeric columns, normalize, and group by a key attribute.

**Note**: We'll reference columns and group-by fields using their Croissant field `@id` for clarity and reproducibility.

In [ ]:
# Pick a numeric field (by @id) for analysis from the field list printed previously.
df = dataframes[main_record_set_id]

# Let's try to heuristically pick numeric fields from the DataFrame
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # Try to convert possible columns to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field_id = col
            break
        except Exception:
            continue
    else:
        raise ValueError("No numeric field found in data.")
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Example threshold
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Heuristically pick a group field (e.g., first non-numeric field)
group_fields = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
if group_fields:
    group_field_id = group_fields[0]
    print(f"\nGrouping by field @{group_field_id}:")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the mean per group (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric_field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped bar plot if group_field is available
if 'group_field_id' in locals():
    plt.figure(figsize=(12,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
We demonstrated how to load a Croissant dataset (`mlcroissant`), enumerate its data structure via `@id`-based referencing, extract the tables, and perform basic exploratory analysis—including filtering, normalization, grouping, and visualizations. 

For further exploration, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/).

_Dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_
